# DepthSR - train + evaluate on Kaggle (GPU T4)
Settings (right panel): **Accelerator = GPU T4 x1**, **Internet = On** (needs a phone-verified Kaggle account).
Then *Run All*. Total time ~1.5-2.5 h (download 10 min, training ~40 min, full evaluation ~1 h).
At the end download `/kaggle/working/depthsr_outputs.zip` and unzip it into your repo folder.

In [ ]:
# 1. Get the code: set REPO_URL to your GitHub repo, or upload the repo zip as a Kaggle dataset
REPO_URL = "https://github.com/krishgarg0o/depth-sr.git"   # <-- change
TEAM = "T74"                                             # <-- your team ID
USNS = "1RUA24CSE0186,1RUA24CSE0197,1RUA24CSE0208,1RUA24CSE0209"                             # <-- your 4 USNs, comma separated
import os, glob, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("depth-sr"):
    if "YOUR_USER" not in REPO_URL:
        !git clone -q $REPO_URL depth-sr
    else:  # fallback: code uploaded as a Kaggle dataset
        src = glob.glob("/kaggle/input/*/depth-sr") or glob.glob("/kaggle/input/*")
        !cp -r {src[0]} depth-sr
os.chdir("/kaggle/working/depth-sr")
!ls

In [ ]:
# 2. Dependencies (ximgproc needs opencv-contrib)
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python 2>/dev/null
!pip install -q opencv-contrib-python-headless h5py
import cv2, torch; print(cv2.__version__, hasattr(cv2, "ximgproc"), torch.cuda.is_available())

In [ ]:
# 3. Download NYU Depth V2 labeled (2.8 GB) and convert -> data/nyu_{train,val,test}.npz
#    If the download fails: add a Kaggle dataset that contains nyu_depth_v2_labeled.mat (Add Input)
mat = glob.glob("/kaggle/input/**/nyu_depth_v2_labeled.mat", recursive=True)
if mat:
    MAT = mat[0]
else:
    MAT = "/kaggle/working/nyu_depth_v2_labeled.mat"
    if not os.path.exists(MAT):
        !wget -q --show-progress -O $MAT http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat
!python scripts/prepare_nyu.py --mat $MAT --out data
!ls -lh data

In [ ]:
# 4. Train TGNet: fp32, lr 2e-4, 6000 iterations, batch 8 (~60 min on a T4)
!python scripts/train.py --data data --iters 6000 --bs 8 --workers 4 --val_every 500

In [ ]:
# 5. Full evaluation on the 449 unseen NYU test images (x2, x4, x8, x16) + noisy protocol + ablation
#    Use --limit 200 if you are short on time.
!python scripts/evaluate.py --dataset nyu --data data --team $TEAM --usns "$USNS" --limit 100 --noisy_limit 30
!cat results/eval_P4_{TEAM}.md

In [ ]:
# 6. Generalisation to an unseen dataset: Middlebury 2005 + 2006 stereo, never used for training.
#    Add Input -> dataset 'minhanhtruong/middleburystereodataset' (the original server refuses automated downloads).
!python scripts/evaluate.py --dataset middlebury --data /kaggle/input --scales 2,4,8 --no_memory --limit 30
!head -60 results/eval_middlebury.md

In [ ]:
# 7. Save a few unseen NYU test images as app examples (16-bit PNG, mm)
import numpy as np
z = np.load("data/nyu_test.npz")
for k, i in enumerate([0, 120, 300]):
    cv2.imwrite(f"samples/nyu{k}_rgb.png", cv2.cvtColor(z["rgb"][i], cv2.COLOR_RGB2BGR))
    cv2.imwrite(f"samples/nyu{k}_depth.png", (z["depth"][i] * 1000).astype(np.uint16))
    lr = cv2.resize(z["depth"][i], (z["depth"][i].shape[1] // 8, z["depth"][i].shape[0] // 8), interpolation=cv2.INTER_CUBIC)
    cv2.imwrite(f"samples/nyu{k}_lowres_x8_depth.png", (lr * 1000).astype(np.uint16))
!ls samples

In [ ]:
# 8. Package everything to download
!zip -qr /kaggle/working/depthsr_outputs.zip checkpoints/tgnet.pt results samples
!ls -lh /kaggle/working/depthsr_outputs.zip
from IPython.display import FileLink; FileLink("/kaggle/working/depthsr_outputs.zip")